In [1]:
%load_ext autoreload
%autoreload 2
import os
import sys
import pickle
from functools import reduce
    
sys.path.append(os.path.abspath("../analysis_tools/"))
from utils import * 

# load first few rows from parquet
import fastparquet

# for median absolute deviation
import scipy.stats as ss

# for Louvain clustering
import community as community_louvain
import networkx as nx
from sklearn.neighbors import NearestNeighbors
import scipy.sparse as sp
import scipy.spatial as spt

# training
from sklearn.metrics import roc_curve, auc, precision_score, recall_score, roc_auc_score, confusion_matrix

# make cell crops
import skimage.io

# for legends heatmap annotations
import matplotlib.lines as mlines
import matplotlib.gridspec as gridspec

# upset plots
from upsetplot import generate_counts, plot
from upsetplot import UpSet


In [ ]:
# Import other datasets
# Anderson data
anderson_data = \
    pd.read_csv('../VISseq_summarydata_v0.2/LMNA_VIS-seq/ExternalData/anderson_2021_variantdata_Formatted.csv')

# Clinvar
clinvar_variants_filterconditions = \
    pd.read_csv('../VISseq_summarydata_v0.2/LMNA_VIS-seq/ExternalData/clinvar_LMNA_missense_ex345_061824_fixed.csv')
condition_map = {'DCM':'DCM', 'Myopathy':'EDMD', 'Lipodystrophy':'FPLD', 'Progeria':'HGPS'}
for disease in ['DCM','Myopathy','Lipodystrophy','Progeria']:
    clinvar_variants_filterconditions[disease] = \
        clinvar_variants_filterconditions["Mapped Conditions"]\
            .map(lambda x: True if condition_map[disease] in x else False)

# Define one way to color variants, use when highlighting specific variants
variant_type_palette = \
    {'Missense':'grey',
     'Synonymous':'darkgreen',
     'WT':'grey', 
     'Frameshift':'purple', 
     '3nt Deletion':'grey', 
     'Nonsense':'purple', 
     'Other':'grey'}
mutation_types=['Missense','Synonymous','Frameshift']

# Variants filtered for spliceAI scores
lmna_variants_spliceAIfiltered = \
    pd.read_csv('../VISseq_summarydata_v0.2/LMNA_VIS-seq/ExternalData/LMNA_AM_CV.csv')


In [ ]:
# Import Gwen's ROC AUC
roc_auc_lmna = pd.read_csv('../VISseq_summarydata_v0.2/LMNA_VIS-seq/AUROC/results/train_results.csv')
roc_auc_lmna = roc_auc_lmna[['aaChanges','test_roc_auc','Example Count']].rename(columns={'aaChanges':'Variant','test_roc_auc':'AUC ROC'})
roc_auc_lmna_controls = \
    pd.read_csv("../VISseq_summarydata_v0.2/LMNA_VIS-seq/AUROC/results-control/train_results.csv")
roc_auc_lmna_controls = roc_auc_lmna_controls[['aaChanges','test_roc_auc']].rename(columns={'aaChanges':'Variant','test_roc_auc':'AUC ROC'})
roc_auc_lmna_replicates = \
    pd.read_csv("../VISseq_summarydata_v0.2/LMNA_VIS-seq/AUROC/results-replicate/train_results.csv").drop(columns='Unnamed: 0').rename(columns={'aaChanges':'Variant','test_roc_auc':'AUC ROC'})


In [4]:
# LMNA information
start_pos = 178
end_pos = 273
LMNA_domain_map = {}
for pos in range(start_pos, end_pos+1):
    if pos < 222:
        LMNA_domain_map[pos] = "Coil 1B"
    elif pos <= 241:
        LMNA_domain_map[pos] = "Linker 12"
    else:
        LMNA_domain_map[pos] = "Coil 2A"
LMNA_domain_colors = \
    {
        'Coil 1B':'goldenrod',
        'Linker 12':'lime',
        'Coil 2A':'deepskyblue'
    }

# Set up for making feature plots
LMNA_WT_nuc_seq = \
    'ATGGAGACCCCGTCCCAGCGGCGCGCCACCCGCAGCGGGGCGCAGGCCAGCTCCACTCCGCTGTCGCCCACCCGCATCACCCGGCTGCAGGAGAAGGAGGACCTGCAGGAGCTCAATGATCGCTTGGCGGTCTACATCGACCGTGTGCGCTCGCTGGAAACGGAGAACGCAGGGCTGCGCCTTCGCATCACCGAGTCTGAAGAGGTGGTCAGCCGCGAGGTGTCCGGCATCAAGGCCGCCTACGAGGCCGAGCTCGGGGATGCCCGCAAGACCCTTGACTCAGTAGCCAAGGAGCGCGCCCGCCTGCAGCTGGAGCTGAGCAAAGTGCGTGAGGAGTTTAAGGAGCTGAAAGCGCGCAATACCAAGAAGGAGGGTGACCTGATAGCTGCTCAGGCTCGGCTGAAGGACCTGGAGGCTCTGCTGAACTCCAAGGAGGCCGCACTGAGCACTGCTCTCAGTGAGAAGCGCACGCTGGAGGGCGAGCTGCATGATCTGCGGGGCCAGGTGGCCAAGCTTGAGGCAGCCCTAGGTGAGGCCAAGAAACAGCTGCAGGACGAAATGCTGCGGCGGGTGGATGCTGAGAACAGGCTGCAGACCATGAAGGAGGAACTGGACTTCCAGAAGAACATCTACAGTGAGGAGCTGCGTGAGACCAAGCGCCGTCATGAGACCCGACTGGTGGAGATTGACAATGGGAAGCAGCGTGAGTTTGAGAGCCGGCTGGCGGATGCGCTGCAGGAACTGCGGGCCCAGCATGAGGACCAGGTGGAGCAGTATAAGAAGGAGCTGGAGAAGACTTATTCTGCCAAGCTGGACAATGCCAGGCAGTCTGCTGAGAGGAACAGCAACCTGGTGGGGGCTGCCCACGAGGAGCTGCAGCAGTCGCGCATCCGCATCGACAGCCTCTCTGCCCAGCTCAGCCAGCTCCAGAAGCAGCTGGCAGCCAAGGAGGCGAAGCTTCGAGACCTGGAGGACTCACTGGCCCGTGAGCGGGACACCAGCCGGCGGCTGCTGGCGGAAAAGGAGCGGGAGATGGCCGAGATGCGGGCAAGGATGCAGCAGCAGCTGGACGAGTACCAGGAGCTTCTGGACATCAAGCTGGCCCTGGACATGGAGATCCACGCCTACCGCAAGCTCTTGGAGGGCGAGGAGGAGAGGCTACGCCTGTCCCCCAGCCCTACCTCGCAGCGCAGCCGTGGCCGTGCTTCCTCTCACTCATCCCAGACACAGGGTGGGGGCAGCGTCACCAAAAAGCGCAAACTGGAGTCCACTGAGAGCCGCAGCAGCTTCTCACAGCACGCACGCACTAGCGGGCGCGTGGCCGTGGAGGAGGTGGATGAGGAGGGCAAGTTTGTCCGGCTGCGCAACAAGTCCAATGAGGACCAGTCCATGGGCAATTGGCAGAtCAAGCGCCAGAATGGAGACGACCCACTGCTCACCTACCGGTTCCCACCAAAGTTCACCCTGAAGGCTGGGCAGGTGGTGACGATCTGGGCTGCAGGAGCTGGGGCCACCCACAGCCCCCCTACCGACCTGGTGTGGAAGGCACAGAACACCTGGGGCTGCGGGAACAGCCTGCGTACGGCTCtCATCAACTCCACTGGGGAAGAAGTGGCCATGCGCAAGCTGGTGCGCTCAGTGACTGTGGTTGAGGACGACGAGGATGAGGATGGAGATGACCTGCTCCATCACCACCACGGCTCCCACTGCAGCAGCTCGGGGGACCCCGCTGAGTACAACCTGCGCTCGCGCACCGTGCTGTGCGGGACCTGCGGGCAGCCTGCCGACAAGGCATCTGCCAGCGGCTCAGGAGCCCAGGTGGGCGGACCCATCTCCTCTGGCTCTTCTGCCTCCAGTGTCACGGTCACTCGCAGCTACCGCAGTGTGGGGGGCAGTGGGGGTGGCAGCTTCGGGGACAATCTGGTCACCCGCTCCTACCTCCTGGGCAACTCCAGCCCCCGAACCCAGAGCCCCCAGAACTGCAGCATCATGTAG'
LMNA_WT_aa_seq = \
    str(Seq(LMNA_WT_nuc_seq).translate())
lmna_pymol_view = \
    '''set_view (\
        -0.915147185,    0.403044611,   -0.007183676,\
        -0.175614133,   -0.414666086,   -0.892865419,\
        -0.362844229,   -0.815845907,    0.450264663,\
         0.000000000,    0.000000000, -382.015167236,\
      -190.245574951, -108.335136414,  109.851905823,\
       301.183654785,  462.846679688,  -20.000000000 )'''
lmna_pymol_view2 = \
    '''set_view (\
     0.071904697,   -0.778118074,   -0.623987436,\
     0.870689154,    0.354146332,   -0.341291487,\
     0.486548126,   -0.518758714,    0.702964723,\
     0.000000000,    0.000000000, -374.783477783,\
    -2.406383991,  -20.413566589,  -11.670532227,\
   295.482147217,  454.084808350,  -20.000000000 )'''


In [ ]:
# Read profiles
df_profiles_merged = pd.read_csv('../VISseq_summarydata_v0.2/LMNA_VIS-seq/Merged/LMNA_averaged_medianplusEMD_010425.csv', index_col=None)
df_profiles_merged.drop(columns='Unnamed: 0',inplace=True)
df_profiles_merged['Variant_Class'] = \
    pd.Categorical(df_profiles_merged['Variant'].astype(str).apply(variant_classification),
                    categories=mutation_types, ordered=True)
features_common = [c for c in df_profiles_merged.columns if c not in ['Variant','Variant_Class',
                                                                      'NCells_R1','NCells_R2',
                                                                      'UMAP1','UMAP2',
                                                                      'Louvain Cluster',
                                                                      'Morphological Impact Score']]

# Read p-values
df_KSpvalues_merged = \
    pd.read_csv('../VISseq_summarydata_v0.2/LMNA_VIS-seq/Merged/LMNA_merged_KSpvalues_010924.csv')
df_KSpvalues_merged.index = df_KSpvalues_merged['Variant']
df_KSpvalues_merged.drop(columns='Variant',inplace=True)
nonblocked_features = df_KSpvalues_merged.columns
nonblocked_features = [f for f in nonblocked_features if ('Puncta' not in f) & ('Line' not in f)]

# Read features
variant_medians_merged = \
    pd.read_csv('../VISseq_summarydata_v0.2/LMNA_VIS-seq/Merged/LMNA_merged_featuremedians_010625.csv', index_col=0)
variant_medians_merged['Variant_Class'] = \
    pd.Categorical(variant_medians_merged['Variant_Class'].map(lambda x: 'Missense' if x=='Single Missense' else x), 
                   categories=mutation_types,
                   ordered=True)
variant_EMD_merged = \
    pd.read_csv('../VISseq_summarydata_v0.2/LMNA_VIS-seq/Merged/LMNA_merged_featureEMD_010925.csv', index_col=0)
variant_EMD_merged['Variant_Class'] = \
    pd.Categorical(variant_EMD_merged['Variant_Class'].map(lambda x: 'Missense' if x=='Single Missense' else x), 
                   categories=mutation_types,
                   ordered=True)

# Get umap coordinates
umap_coords = \
    df_profiles_merged[['Variant','UMAP1','UMAP2']]

# Highlight biochemical controls
punctuate_controls = ['N195K','L248P','E203G','R189P']
non_punctuate_controls = ['D192V', 'R189W', 'E203K', 'E203V', 'E262K', 'E202K', 'R190Q']
total_controls = punctuate_controls + non_punctuate_controls
umap_coords['Highlight'] = \
    umap_coords['Variant'].map(lambda x: True if x in total_controls else False)
umap_coords['HighlightClass'] = \
    umap_coords['Variant'].map(lambda x: 'Punctuate' if x in punctuate_controls \
                                                     else 'Non-Punctuate' if x in non_punctuate_controls \
                                                     else 'WT' if x == 'WT' \
                                                     else np.nan)



In [6]:
len(features_common)

332

In [ ]:
# Import median or EMD-only profiles and merge them
# R1
df_profiles_R1 = \
    pd.read_csv('../VISseq_summarydata_v0.2/LMNA_VIS-seq/Replicate1/R1_variant_profiles_medianplusEMD_selected_nopunctaline_010425.csv')
df_profiles_R1.drop(columns='Unnamed: 0',inplace=True)
df_medianprofiles_R1 = \
    pd.read_csv('../VISseq_summarydata_v0.2/LMNA_VIS-seq/Replicate1/LMNAT3R1.medianonly.selected.csv')
df_EMDprofiles_R1 = \
    pd.read_csv('../VISseq_summarydata_v0.2/LMNA_VIS-seq/Replicate1/LMNAT3R1.EMDonly.selected.csv')
#df_medianprofiles_R1.drop(columns='Unnamed: 0',inplace=True)
#df_EMDprofiles_R1.drop(columns='Unnamed: 0', inplace=True)

feat_median_selected_R1 = [s for s in df_medianprofiles_R1.columns\
                                if s not in ['Variant',
                                             'Variant_Class',
                                             'Metadata_Object_Count',
                                             'UMAP1',
                                             'UMAP2']
                          ]
feat_EMD_selected_R1    = [s for s in df_EMDprofiles_R1.columns\
                                if s not in ['Variant',
                                             'Variant_Class',
                                             'Metadata_Object_Count',
                                             'UMAP1',
                                             'UMAP2']
                          ]
df_medianprofiles_R1.index = df_profiles_R1.Variant
df_EMDprofiles_R1.index = df_profiles_R1.Variant

# R2
df_profiles_R2 = \
    pd.read_csv('../VISseq_summarydata_v0.2/LMNA_VIS-seq/Replicate2/R2_variant_profiles_medianplusEMD_selected_nopunctaline_010425.csv')
df_profiles_R2.drop(columns='Unnamed: 0',inplace=True)
df_medianprofiles_R2 = \
    pd.read_csv('../VISseq_summarydata_v0.2/LMNA_VIS-seq/Replicate2/LMNAT3R2.medianonly.selected.csv')
df_EMDprofiles_R2 = \
    pd.read_csv('../VISseq_summarydata_v0.2/LMNA_VIS-seq/Replicate2/LMNAT3R2.EMDonly.selected.csv')
#df_medianprofiles_R2.drop(columns='Unnamed: 0',inplace=True)
#df_EMDprofiles_R2.drop(columns='Unnamed: 0', inplace=True)

feat_median_selected_R2 = [s for s in df_medianprofiles_R2.columns\
                                if s not in ['Variant',
                                             'Variant_Class',
                                             'Metadata_Object_Count',
                                             'UMAP1',
                                             'UMAP2']
                          ]
feat_EMD_selected_R2    = [s for s in df_EMDprofiles_R2.columns\
                                if s not in ['Variant',
                                             'Variant_Class',
                                             'Metadata_Object_Count',
                                             'UMAP1',
                                             'UMAP2']
                          ]
df_medianprofiles_R2.index = df_profiles_R2.Variant
df_EMDprofiles_R2.index = df_profiles_R2.Variant

# merge by taking average
feat_median_common = list(set(feat_median_selected_R1).intersection(set(feat_median_selected_R2)))
feat_EMD_common    = list(set(feat_EMD_selected_R1).intersection(set(feat_EMD_selected_R2)))
var_common = list(set(df_profiles_R1.Variant).intersection(set(df_profiles_R2.Variant)))

df_medianprofiles_merged = \
    (df_medianprofiles_R1.loc[var_common,feat_median_common] + df_medianprofiles_R2.loc[var_common,feat_median_common])/2 #Average profiles
df_EMDprofiles_merged = \
    (df_EMDprofiles_R1.loc[var_common,feat_EMD_common] + df_EMDprofiles_R2.loc[var_common,feat_EMD_common])/2 #Average profiles
df_medianprofiles_merged = df_medianprofiles_merged.reset_index()
df_EMDprofiles_merged = df_EMDprofiles_merged.reset_index()

df_medianprofiles_merged['Variant_Class'] = \
    pd.Categorical(df_medianprofiles_merged['Variant'].astype(str).apply(variant_classification),
                       categories=mutation_types, ordered=True)
df_EMDprofiles_merged['Variant_Class'] = \
    pd.Categorical(df_EMDprofiles_merged['Variant'].astype(str).apply(variant_classification),
                       categories=mutation_types, ordered=True)

# Compute impact scores for median and EMD merged profiles, and save
df_medianprofiles_merged_synmedian = \
    df_medianprofiles_merged\
        .query('Variant_Class == "Synonymous"')\
        [feat_median_common].agg(np.median)
variant_cosine_similarity_median = \
    cosine_similarity(df_medianprofiles_merged[feat_median_common], 
                      df_medianprofiles_merged_synmedian.values.reshape(1,-1))
df_medianprofiles_merged['Morphological Impact Score'] = \
    (1-variant_cosine_similarity_median[:,0])/2
#df_medianprofiles_merged.to_csv('./consensus_profiles/LMNA_merged_medianprofiles_121425.csv')

df_EMDprofiles_merged_synmedian = \
    df_EMDprofiles_merged\
        .query('Variant_Class == "Synonymous"')\
        [feat_EMD_common].agg(np.median)
variant_cosine_similarity_EMD = \
    cosine_similarity(df_EMDprofiles_merged[feat_EMD_common], 
                      df_EMDprofiles_merged_synmedian.values.reshape(1,-1))
df_EMDprofiles_merged['Morphological Impact Score'] = \
    (1-variant_cosine_similarity_EMD[:,0])/2
#df_EMDprofiles_merged.to_csv('./consensus_profiles/LMNA_merged_EMDprofiles_121425.csv')


In [11]:
# Import information about features
# Annotations
feature_df = pd.read_csv('./consensus_plots_postreview/feature_df.csv')
feature_df.index = feature_df['feature']
feature_df = feature_df.rename(columns={'feature.1':'feature'})

# Selected feature clustering
selected_feature_medianclustering = pd.read_csv('./consensus_plots_postreview/selected_feature_medianonly_reproducible_clustering_122825.csv', index_col=0)
selected_feature_EMDclustering = pd.read_csv('./consensus_plots_postreview/selected_feature_EMDonly_reproducible_clustering_122825.csv', index_col=0)


In [7]:
# Pull single cell feature information to find cells - - need single cell image + feature data for this portion!
# features_to_pull = \
#     [
#         'Mean_NucleiExpanded_Intensity_MeanIntensity_CH1',
#         'Mean_Nuclei_AreaShape_FormFactor',
#         'Mean_NuclearBoundary_Intensity_MeanIntensity_CH1',
#         'Mean_NucleiExpanded_Granularity_1_CH1',
#         'Mean_Nuclei_AreaShape_Eccentricity',
#         'Mean_Nuclei_AreaShape_Solidity',
#         'Mean_NucleiExpanded_Correlation_Overlap_CH0_CH1',
#         'Mean_NucleiExpanded_RadialDistribution_MeanFrac_CH0_8of10',
#         'Mean_NucleiExpanded_RadialDistribution_RadialCV_CH1_9of10',
#         'Mean_NuclearBoundary_Correlation_Manders_CH1_CH0',
#         'Mean_Nuclei_Neighbors_SecondClosestDistance_10',
#         'Mean_Nuclei_Neighbors_FirstClosestDistance_10',
#         'Mean_NucleiExpanded_Granularity_7_CH1',
#         'Mean_NucleiExpanded_Granularity_8_CH1',
#         'Mean_NucleiExpanded_RadialDistribution_RadialCV_CH0_8of10',
#         'Mean_NucleiExpanded_RadialDistribution_RadialCV_CH0_10of10',
#         'Mean_NuclearBoundary_Correlation_RWC_CH1_CH0',
#         'Mean_NucleiExpanded_RadialDistribution_FracAtD_CH0_1of10',
#     ]
landmark_features = \
    [
        'Mean_NucleiExpanded_Intensity_MeanIntensity_CH1',
        'Mean_Nuclei_AreaShape_FormFactor',
        'Mean_NuclearBoundary_Intensity_MeanIntensity_CH1',
        'Mean_NucleiExpanded_Granularity_1_CH1'
    ]
# highly_interpretable_features = \
#     feature_df.loc[((feature_df["Method"].astype(str).isin(["AreaShape","Correlation"])) | \
#          (feature_df["Imaging Channel"].astype(str) == "mEGFP-\nLaminA") & \
#          (feature_df["Method"].astype(str).isin(["Intensity","RadialDistribution","Granularity"])) # Shape, Correlation + LMNA intensity, radial distribution, granularity features
#     )].index
features_to_pull = \
    reduce(np.union1d, [
        np.asarray(landmark_features), 
        #selected_feature_medianclustering['feature'].values, 
        #selected_feature_EMDclustering['feature'].str.removesuffix('_EMD').values,
        #highly_interpretable_features
                       ])


#r1_phenotyping_path = '/net/fowler/vol1/shared/fisseq/PBv2b_T3R1/phenotyping/'
r2_phenotyping_path = '/net/fowler/vol1/shared/fisseq/PBv2b_T3R2/phenotyping/'

# Import cell information
# genotypes_df_r1 = \
#     pd.read_parquet('./R1_features/LMNAT3R1.cellprofiler_122824.cells_full.parquet', 
#                     columns=['upBarcode',
#                              'aaChanges',
#                              'editDistance',
#                              'AreaShape_Center_X',
#                              'AreaShape_Center_Y',
#                              'tile_x',
#                              'tile_y',
#                              'tile_index',
#                              'well']+\
#                             features_to_pull
#                    )
# genotypes_df_r1 = \
#     genotypes_df_r1\
#        .assign(Replicate=1)\
#        .query('editDistance in [0,1]')
# genotypes_df_r1['PhenotypePath'] = \
#     genotypes_df_r1.apply(lambda row: construct_phenotype_path(r1_phenotyping_path,
#                                                                row['well'],
#                                                                row['tile_x'],
#                                                                row['tile_y']), 
#                           axis=1)
# genotypes_df_r1['DAPIPath'] = \
#     genotypes_df_r1.apply(lambda row: construct_phenotype_path(r1_phenotyping_path,
#                                                                row['well'],
#                                                                row['tile_x'],
#                                                                row['tile_y'],
#                                                                filename='channel0.tif'), 
#                           axis=1)
# genotypes_df_r1['CellMaskPath'] = \
#     genotypes_df_r1.apply(lambda row: construct_phenotype_path(r1_phenotyping_path,
#                                                                row['well'],
#                                                                row['tile_x'],
#                                                                row['tile_y'],
#                                                                filename='cells.tif'), 
#                           axis=1)

genotypes_df_r2 = \
    pd.read_parquet('/net/fowler/vol1/shared/fisseq/PBv2b_T3R2/output/wellall_seqgrid5_phenogrid20.cellprofiler_122824.cells_full.parquet', 
                    columns=['Unnamed: 0.1',
                             'upBarcode',
                             'aaChanges',
                             'editDistance',
                             'AreaShape_Center_X',
                             'AreaShape_Center_Y',
                             'tile_x',
                             'tile_y',
                             'tile_index',
                             'well']+\
                            list(features_to_pull)
                            #nonblocked_features
                   )
# genotypes_df_r2 = \
#     genotypes_df_r2\
#        .assign(Replicate=2)\
#        .query('editDistance in [0,1]')
genotypes_df_r2['CellCropsPath'] = \
    genotypes_df_r2.apply(lambda row: construct_phenotype_path_v2(r2_phenotyping_path,
                                                                  row['well'],
                                                                  row['tile_x'],
                                                                  row['tile_y'],
                                                                  filename='cell_images_100.tif'), 
                          axis=1)
genotypes_df_r2['CellMasksPath'] = \
    genotypes_df_r2.apply(lambda row: construct_phenotype_path_v2(r2_phenotyping_path,
                                                                  row['well'],
                                                                  row['tile_x'],
                                                                  row['tile_y'],
                                                                  filename='mask_images_100.tif'), 
                          axis=1)

# Set well index and crop index in tile
genotypes_df_r2.rename(columns={'Unnamed: 0.1':'CellIndex'}, inplace=True)
genotypes_df_r2['CropIndex'] = \
    genotypes_df_r2.groupby(["tile_index", "well"]).cumcount()

#genotypes_df = pd.concat([genotypes_df_r1,genotypes_df_r2], axis=0)


In [12]:
# Plot UMAP with Anderson controls
# ------------------------------------
# Data setup
# ------------------------------------
variants_assayed_umap_toplot = df_profiles_merged.copy()
variants_assayed_umap_toplot = pd.merge(
    variants_assayed_umap_toplot.reset_index(drop=True),
    anderson_data,
    on='Variant',
    how='left'
)

variantssorted_bypunctuate = (
    anderson_data[['Variant','HEK']]
    .drop_duplicates()
    .sort_values(by='HEK')['Variant']
    .values
)
variantssorted_bypunctuate = variantssorted_bypunctuate[variantssorted_bypunctuate != 'WT']
variants_assayed_umap_toplot = variants_assayed_umap_toplot.sort_values(by='Variant')
variant_anno = (
    variants_assayed_umap_toplot
    .query('Variant_Class == "Missense"')
    .query('~`HEK`.isna()')
)

# Set punctate threshold
punctate_percent_threshold = 15 #punctate threshold
variant_anno['punctate_true'] = \
    variant_anno['HEK'].map(lambda x: True if x>= punctate_percent_threshold else False)

# ------------------------------------
# Plotting
# ------------------------------------
fig = plt.figure(figsize=(5,4.7))
gs = gridspec.GridSpec(1, 2, width_ratios=[1, 0.05], wspace=0.05)
ax = fig.add_subplot(gs[0])
cax = fig.add_subplot(gs[1])

# --- Plotting the data ---
# Background (greyed out) points:
sns.scatterplot(
    data=variants_assayed_umap_toplot,
    x='UMAP1',
    y='UMAP2',
    color='grey',
    alpha=0.3,
    s=40,
    ax=ax,
    legend=False
)

# Highlighted (Anderson) points, positive first, then negative:
sns.scatterplot(
    data=variant_anno\
            .rename(columns={'HEK': 'Anderson et al. (2021)\nPercent Punctate'}),
    x='UMAP1',
    y='UMAP2',
    hue='Anderson et al. (2021)\nPercent Punctate',
    palette="viridis",
    s=300,
    style="punctate_true",
    markers={True:'P', False:'X'},
    ax=ax,
    legend=False
)

# Set up the ScalarMappable for the colorbar:
norm = mcolors.Normalize(vmin=0, vmax=50)
sm = plt.cm.ScalarMappable(cmap="viridis", norm=norm)
sm.set_array([])

# Create the colorbar in the reserved axis:
cbar = plt.colorbar(sm, cax=cax, extend="both", spacing="proportional",
                     boundaries=np.linspace(0, 50))
cbar.ax.set_yscale('linear')
cbar.set_ticklabels(ticklabels=[0,10,20,30,40,50],fontsize=14)

# Final formatting of the main axes:
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel('UMAP 1', fontsize=16)
ax.set_ylabel('UMAP 2', fontsize=16)
plt.tight_layout()
plt.close()

# ------------------------------------
# (Optional) Save the figure
# ------------------------------------
fig.savefig('./consensus_plots_postreview/LMNAT3.UMAP.andersonvariants.011025.pdf', dpi=600)


In [13]:
# Plot against clinvar designation
clinvar_variants_filtersplice = \
    clinvar_variants_filterconditions\
        .rename(columns={'aaChanges':'Variant'})\
        .query('Variant in @lmna_variants_spliceAIfiltered["protein_variant"]')
variants_assayed_umap_toplot = \
    df_profiles_merged
variants_assayed_umap_toplot = pd.merge(variants_assayed_umap_toplot, 
                                        clinvar_variants_filtersplice\
                                            [['Variant','Variant Designation','Mapped Conditions']],
                                        on='Variant', 
                                        how='left')
variant_anno = \
    variants_assayed_umap_toplot\
        .query('Variant_Class == "Missense"')\
        .query('~`Variant Designation`.isna()')
    
fig,ax=plt.subplots(figsize=(4,4))
sns.scatterplot(data=variants_assayed_umap_toplot, 
                x='UMAP1',
                y='UMAP2', 
                #hue='Variant_Class', 
                color='grey',
                alpha=0.3,
                #palette=variant_type_palette,
                s=40,
                ax=ax,
                legend=False)
variant_anno['zorder'] = \
    variant_anno['Variant Designation'].map({'VUS':0,'B/LB':1,'P/LP':2})
variant_anno = variant_anno.sort_values(by='zorder')
sns.scatterplot(data=variant_anno\
                    .rename(columns={'Variant Designation':'ClinVar'}),
                x='UMAP1',
                y='UMAP2', 
                hue='ClinVar', 
                palette={'VUS':'yellow',
                         'B/LB':'blue',
                         'P/LP':'red'},
                s=300,
                marker="^",
                ax=ax
               )

# Remove x-axis tick labels and ticks
plt.xticks([])

# Remove y-axis tick labels and ticks
plt.yticks([])

# Set legend info
plt.legend(fontsize=14, title='', title_fontsize=16)

# Label x and y-axes
plt.xlabel('UMAP 1', fontsize=16)
plt.ylabel('UMAP 2', fontsize=16)

plt.tight_layout()
plt.close()

fig.savefig('./consensus_plots_postreview/LMNAT3.UMAP.clinvarvariants.011025.pdf', dpi=600)


In [14]:
# Plot grey Missense points first (in the back) with lower alpha,
# synonymous and frameshift points larger, and P/LP variants on top

# Split the data into grey points and other points
subset_grey = df_profiles_merged[df_profiles_merged['Variant_Class'] == 'Missense']
subset_color = df_profiles_merged[df_profiles_merged['Variant_Class'] != 'Missense']
variant_anno = pd.merge(subset_grey, 
                        clinvar_variants_filtersplice\
                            [['Variant','Variant Designation','Mapped Conditions']]\
                            .query('`Variant Designation`.isin(["P","LP","P/LP"])'),
                        on='Variant', 
                        how='inner')


# Create the figure and axis
fig, ax = plt.subplots(figsize=(4, 4))

sns.scatterplot(data=subset_grey, 
                x='UMAP1', 
                y='UMAP2', 
                color='grey', 
                s=40, 
                alpha=0.3,
                ax=ax,
                legend=False)

# Plot the colored points on top
sns.scatterplot(data=subset_color, 
                x='UMAP1', 
                y='UMAP2', 
                hue='Variant_Class',
                palette=variant_type_palette,
                s=40,
                ax=ax,
                legend=False)

# Plot path variants
sns.scatterplot(data=variant_anno\
                    .rename(columns={'Variant Designation':'ClinVAR'}),
                x='UMAP1',
                y='UMAP2', 
                hue='ClinVAR', 
                palette={'P/LP':'red'},
                s=150,
                marker="^",
                ax=ax)

# Create custom legend entries
syn = mlines.Line2D([], [], color='green', marker='o', linestyle='',
                    markersize=7, label='Synonymous')
sm = mlines.Line2D([], [], color='grey', marker='o', linestyle='',
                    markersize=7, alpha=0.3, label='Missense')
fs = mlines.Line2D([], [], color='purple', marker='o', linestyle='',
                   markersize=7, label='Frameshift')

# Add the custom legend
# first legend: variant types
legend1 = ax.legend(handles=[syn, sm, fs],
                    markerscale=1,
                    title='Variant Type',
                    loc='upper right',
                    fontsize=10,
                    title_fontsize=11)
ax.add_artist(legend1)

# **second handle for ClinVar P/LP**
clinvar = mlines.Line2D([], [], color='red', marker='^', linestyle='',
                        markersize=8, label='ClinVar P/LP')

# **second legend: ClinVar**
ax.legend(handles=[clinvar],
          loc='lower right',
          fontsize=10,
          title_fontsize=12)

# Remove x-axis and y-axis tick labels and ticks
plt.xticks([])
plt.yticks([])

# Label axes
plt.xlabel('UMAP 1', fontsize=16)
plt.ylabel('UMAP 2', fontsize=16)
plt.tight_layout()

# Save the figure
fig.savefig('./consensus_plots_postreview/LMNAT3.UMAP.clinvarpath.053025.pdf', dpi=600)
plt.close()


In [15]:
# Plot impact as a function of variant class, including anderson controls and clinVAR data
# merge with anderson controls data
df_toplot = df_profiles_merged.merge(roc_auc_lmna, on='Variant', how='inner').copy()
df_toplot = pd.merge(
    df_toplot.reset_index(drop=True),
    anderson_data,
    on='Variant',
    how='left'
)
punctate_label='Aggreg.\nControls'

# merge with clinVAR
# Remove single clinvar benign
clinvar_variants_filtersplice = \
    clinvar_variants_filterconditions\
        .rename(columns={'aaChanges':'Variant'})\
        .query('Variant in @lmna_variants_spliceAIfiltered["protein_variant"]')
clinvar_variants_filtersplice_benignremoved = \
    clinvar_variants_filtersplice\
        .query('`Variant Designation` != "B/LB"')
df_toplot2 = pd.merge(df_toplot, 
                      clinvar_variants_filtersplice_benignremoved\
                         [['Variant','Variant Designation','Mapped Conditions']],
                      on='Variant', 
                      how='left')
df_toplot2 = \
    df_toplot2.rename(columns={'Variant Designation':'ClinVAR Label'})
df_toplot3 = df_toplot2.copy()

# Core variant‐class categories
df_toplot3['Synonymous']   = df_toplot3['Variant_Class'] == 'Synonymous'
df_toplot3['Missense']     = df_toplot3['Variant_Class'] == 'Missense'
df_toplot3['Frameshift']   = df_toplot3['Variant_Class'] == 'Frameshift'

# Punctate control
df_toplot3[punctate_label] = df_toplot3['HEK'] >= punctate_percent_threshold

# ClinVAR categories from your merged clinvar table
df_toplot3['ClinVAR VUS']  = df_toplot3['ClinVAR Label'] == 'VUS'
df_toplot3['ClinVAR P/LP'] = df_toplot3['ClinVAR Label'].isin(['P','LP','P/LP'])

# Melt
id_vars    = ['Variant','Morphological Impact Score','AUC ROC']
value_vars = [
    'Synonymous','Missense','Frameshift',
    punctate_label,'ClinVAR VUS','ClinVAR P/LP'
]

df_long = df_toplot3.melt(
    id_vars=id_vars,
    value_vars=value_vars,
    var_name='Category',
    value_name='InCategory'
)

# keep only the True rows
df_long = df_long[df_long['InCategory']]

# Plot figure
fig, ax = plt.subplots(figsize=(5,4))
boxplot_with_significance(
    df=df_long,
    x_col='Category',
    y_col='Morphological Impact Score',
    hue_col='Category',  # same as x_col
    order=['Synonymous','Missense','Frameshift',punctate_label,'ClinVAR VUS','ClinVAR P/LP'],
    palette={'Synonymous':'darkgreen',
             'Missense':'grey',
             'Frameshift':'purple',
             punctate_label:'#43C59E',
             'ClinVAR VUS':'yellow',
             'ClinVAR P/LP':'red'},
    alpha=0.001,    # Show significance if p < 0.001
    pairs=[("Synonymous",'Missense'),
           ('Missense',"Frameshift"),
           ("Synonymous",punctate_label),
           ('Missense',punctate_label),
           ("Synonymous","ClinVAR P/LP"),
           ("ClinVAR VUS","ClinVAR P/LP")],
    offset=0.02,
    xlabels=["Synon","Missense","Frame-\nshift",punctate_label,"VUS","P/LP"],
    xlabel_fontsize=10.5,
    ax=ax,
    show_points=True,
    anno_sample_sizes=True
)

plt.ylim(0, 1)
plt.ylabel('Impact Score', fontsize=12)
plt.xlabel('')
ax.tick_params(axis='y', labelsize=10.5)
plt.tight_layout()
plt.close()
fig.savefig('./consensus_plots_postreview/morphological_impact.variantclass_punctate_clinVAR.053025.pdf', dpi=600)


In [19]:
# Add on information from medians, EMDs and make one big Impact Score plot
df_long_merged = \
    df_long.rename(columns={'Morphological Impact Score':'Both'})\
        .merge(df_medianprofiles_merged[['Variant','Morphological Impact Score']]\
                    .rename(columns={'Morphological Impact Score':'Median only'}),
                    on='Variant', how='left')\
        .merge(df_EMDprofiles_merged[['Variant','Morphological Impact Score']]\
                    .rename(columns={'Morphological Impact Score':'EMD only'}),
                    on='Variant', how='left')\

value_vars = ['Both','Median only','EMD only']
df_long_all = df_long_merged.melt(
    id_vars=['Variant', 'Category'],
    value_vars=value_vars,
    var_name='Feature Summarization',
    value_name='Morphological Impact Score'
)

# Plot figure
mis_auroc_palette = \
    {
        'Both':'#83adb5',
        'Median only':'#c7bbc9',
        'EMD only':'#5e3c58',
        'AUC ROC':'#2e4045'
    }
fig, ax = plt.subplots(figsize=(7,4))
sns.boxplot(
    data=df_long_all,
    x='Category',
    y='Morphological Impact Score',
    hue='Feature Summarization',
    fill=True,
    showfliers=False,
    palette=mis_auroc_palette,
    boxprops={"alpha": 0.4},
    ax=ax
)
sns.stripplot(
    data=df_long_all,
    x='Category',
    y='Morphological Impact Score',
    hue='Feature Summarization',
    dodge=True,
    #order=order,
    palette=mis_auroc_palette,
    jitter=True,
    alpha=0.7,
    linewidth=0.5,
    size=2,
    ax=ax,
    zorder=1.5,
    legend=False
)
plt.ylim(0, 1)
plt.ylabel('Impact Score', fontsize=12)
ax.set_xticklabels(["Synon","Missense","Frame-\nshift",punctate_label,"VUS","P/LP"], fontsize=10.5)
plt.xlabel('')
ax.tick_params(axis='y', labelsize=10.5)
ax.legend(loc=[0.34,0.1])
plt.tight_layout()
plt.close()
fig.savefig('./consensus_plots_postreview/morphological_impact.variant_class.bothVSmedianVSemd.121625.pdf', dpi=600)


In [21]:
# Plot impact as a function of variant class, including anderson controls
# add on controls
wt_controls_name = 'WT Samples'
roc_auc_lmna_controls['Category'] = wt_controls_name
df_long_pluscontrols = \
    pd.concat([df_long[['Variant','AUC ROC','Category']], roc_auc_lmna_controls])

# Plot figure
color_palette_toplot = \
    {
        wt_controls_name:'palegreen',
        'Synonymous':'darkgreen',
        'Missense':'grey',
        'Frameshift':'purple',
        punctate_label:'#43C59E',
        'ClinVAR VUS':'yellow',
        'ClinVAR P/LP':'red'
    }
fig, ax = plt.subplots(figsize=(5.5,4))
boxplot_with_significance(
    df=df_long_pluscontrols,
    x_col='Category',
    y_col='AUC ROC',
    hue_col='Category',  # same as x_col
    order=list(color_palette_toplot.keys()),
    palette=color_palette_toplot,
    alpha=0.001,    # Show significance if p < 0.001
    pairs=[("Synonymous",'Missense'),
           ('Missense',"Frameshift"),
           ("Synonymous",punctate_label),
           ('Missense',punctate_label),
           ("Synonymous","ClinVAR P/LP"),
           ("ClinVAR VUS","ClinVAR P/LP")],
    xlabels=["WT\nSamples",
             "Synon","Missense","Frame-\nshift",
             punctate_label,"VUS","P/LP"],
    xlabel_fontsize=10.5,
    offset=0.0125,
    ax=ax,
    show_points=True,
    anno_sample_sizes=True
)

plt.ylim(0.45, 1)
plt.ylabel('Distinguishability Score', fontsize=12)
plt.xlabel('')
ax.tick_params(axis='y', labelsize=10.5)
plt.tight_layout()
plt.close()
fig.savefig('./consensus_plots_postreview/aucroc.variantclass_punctate_clinVAR.053025.pdf', dpi=600)


In [23]:
# Get AUC ROC from LP/P vs Synonymous
# Subset to the two clinical categories
# List of (positive, negative) Category pairs
comparisons = [
    ("ClinVAR P/LP", "Synonymous"),
    (punctate_label,  "Synonymous")
]
features_to_use = ['Both','Median only','EMD only','AUC ROC']
names_of_categories = \
    {
        "ClinVAR P/LP":"ClinVAR P/LP",
        punctate_label:"Aggreg. Controls",
        "Synonymous":"Synon"
    }
names_of_features = \
    {
        'Both':'Impact (Both)',
        'Median only':'Impact (Median)',
        'EMD only':'Impact (EMD)',
        'AUC ROC':'Distinguishability'
    }

# Create side-by-side subplots
fig, axes = plt.subplots(ncols=2, figsize=(6,3), sharey=True)

for ax, (pos, neg) in zip(axes, comparisons):
    # Filter to just the two categories, and only rows where InCategory == True
    df_sub = df_long_merged[
        df_long_merged['Category'].isin([pos, neg]) &
        df_long_merged['InCategory']
    ].copy()
    
    # Build true labels (1 for positive class, 0 for negative) and scores
    y_true   = (df_sub['Category'] == pos).astype(int)
    for j, feat in enumerate(features_to_use):
        y_scores = df_sub[feat]
        
        # Compute ROC curve and AUC
        fpr, tpr, _ = roc_curve(y_true, y_scores)
        roc_auc     = auc(fpr, tpr)
        
        # Plot ROC
        ax.plot(fpr, tpr, label=f"{names_of_features[feat]} {roc_auc:.2f}", color=mis_auroc_palette[feat])
    ax.plot([0, 1], [0, 1], linestyle="--", color="grey", linewidth=0.8)
    ax.set_xticks([0,1])
    ax.set_yticks([0,1])
    
    # Formatting
    ax.set_title(f"{names_of_categories[pos]} vs {names_of_categories[neg]}", fontsize=12)
    ax.set_xlabel("FPR", fontsize=12)
    if pos == "ClinVAR P/LP":
        ax.set_ylabel("TPR", fontsize=12)
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.legend(loc="lower right", fontsize=8)

plt.tight_layout()
fig.savefig('./consensus_plots_postreview/LMNAT3.clinvarPvsSynon.ROCcurves.061025.pdf')
plt.close()


In [29]:
# Plot ROC AUC as a function of number of training examples
df_toplot = \
    df_profiles_merged\
        .merge(roc_auc_lmna_replicates.query('replicate==1')[['AUC ROC','Variant']].rename(columns={'AUC ROC':'AUC ROC Replicate 1'}), 
               on='Variant', how='inner')\
        .merge(roc_auc_lmna_replicates.query('replicate==2')[['AUC ROC','Variant']].rename(columns={'AUC ROC':'AUC ROC Replicate 2'}), 
               on='Variant', how='inner')

# Compute linear regression parameters
r1_auc = df_toplot.sort_values(by='Variant_Class')['AUC ROC Replicate 1']
r2_auc = df_toplot.sort_values(by='Variant_Class')['AUC ROC Replicate 2']

slope, intercept, r_value, p_value, std_err = ss.linregress(r1_auc, r2_auc)

# Prepare line values for the best-fit line
line_x = np.linspace(r1_auc.min(), r2_auc.max(), 100)
line_y = slope * line_x + intercept

# Plot
fig, ax = plt.subplots(figsize=(5.2, 5))
sns.scatterplot(
    data=df_toplot.sort_values(by='Variant_Class'),
    x='AUC ROC Replicate 1',
    y='AUC ROC Replicate 2',
    hue='Variant_Class',
    palette=variant_type_palette,
    ax=ax,
    legend=False
)
#ax.plot(line_x, line_y, color='red', label='Best Fit Line')

# Add labels
ax.set_xlabel('Distinguishability Score Replicate 1', fontsize=14)
ax.set_ylabel('Distinguishability Score Replicate 2', fontsize=14)
ax.set_xlim(0.4,1)
ax.set_ylim(0.4,1)

# Display R^2 on the plot
ax.text(
    0.05, 0.9, 
    f'$R = {r_value:.2f}$', 
    transform=ax.transAxes, 
    fontsize=20, 
    bbox=dict(facecolor='white', alpha=0.5)
)

# Show legend and plot
# ax.legend(loc='lower right', markerscale=2, title='Variant Type', title_fontsize=14, fontsize=12)
# ax.tick_params(labelsize=12)
plt.tight_layout()
fig.savefig('./consensus_plots_postreview/LMNAT3.AUROC.correlationbetweenreps.061325.pdf')
plt.close()

# Plot ROC AUC as a function of number of training examples
graph_auc_examples(
    df_profiles_merged.merge(roc_auc_lmna, on='Variant', how='inner').sort_values(by='Variant_Class'),
    img_save_path = "./consensus_plots_postreview/LMNAT3.auc_roc.trainingexamples.061125.pdf",
    log_axis = True,
    xlim = (200,10000),
    y_label = "Distinguishability Score"
)
plt.close()


In [31]:
# Relabel clusters
cluster_remap = \
    {
        0: 8,
        1: 9,
        2: 2,
        3: 10,
        4: 3,
        5: 4,
        6: 6,
        7: 7,
        8: 5,
        9: 1
    }
df_profiles_merged['Cluster'] = \
    df_profiles_merged['Louvain Cluster'].map(lambda x: cluster_remap[x])


In [33]:
# Heatmap of landmark features over clusters
nclust=np.max(df_profiles_merged['Cluster'].values)
landmark_features = \
    ['Mean_NucleiExpanded_Intensity_MeanIntensity_CH1',
     'Mean_NuclearBoundary_Intensity_MeanIntensity_CH1',
     'Mean_NucleiExpanded_Granularity_1_CH1',
     'Mean_Nuclei_AreaShape_FormFactor',
    ]
feature_rename = \
    {
     'Mean_NucleiExpanded_Intensity_MeanIntensity_CH1':'Lamin A Nuclear\nIntensity',
     'Mean_NuclearBoundary_Intensity_MeanIntensity_CH1':'Lamin A Nuclear\nBoundary Intensity',
     'Mean_NucleiExpanded_Granularity_1_CH1':'Lamin A Nuclear\nGranularity 1',
     'Mean_Nuclei_AreaShape_FormFactor': 'Nuclear Shape\nFactor',
    }
variant_class_palette = \
    {(i+1):sns.color_palette('tab10')[i] for i in range(nclust)}
variant_medianEMD_landmark_merged_withcluster = \
    variant_medians_merged[['Variant']+landmark_features]\
        .rename(columns={x:feature_rename[x]+' median' for x in landmark_features})\
        .merge(variant_EMD_merged[['Variant']+landmark_features]\
                   .rename(columns={x:feature_rename[x]+' EMD' for x in landmark_features}), 
               on='Variant')\
        .merge(df_profiles_merged[['Variant','Cluster']], on='Variant')
total_landmark_features = \
    [feature_rename[c]+' median' for c in landmark_features] + \
    [feature_rename[c]+' EMD' for c in landmark_features]
medianvalues_landmark_features_bycluster = \
    variant_medianEMD_landmark_merged_withcluster\
        [total_landmark_features+['Cluster']]\
        .groupby('Cluster')\
        .agg(np.median)

landmark_clustermap = \
    sns.clustermap(medianvalues_landmark_features_bycluster.T,
                   vmin=-6,
                   vmax=6,
                   row_cluster=False,
                   #metric="correlation",
                   #method="ward",
                   col_colors=[variant_class_palette[x] \
                                   for x in medianvalues_landmark_features_bycluster.index],
                   xticklabels=True,
                   yticklabels=False,
                   figsize=(4,4),
                   linewidths=.25,
                   cmap='bwr',
                   cbar_pos=(0.05, 0.15, 0.03, 0.6),
                   cbar_kws={'extend':'both', 'shrink': 0.5, 'pad': 0.001}
                  )
landmark_clustermap.ax_heatmap.set_xlabel('Cluster', fontsize=18)

# After creating the cluster map change annotation sizes
pos = landmark_clustermap.ax_col_colors.get_position()
pos_d = landmark_clustermap.ax_col_dendrogram.get_position()

# Make anno/dendro 2 times taller and move it upwards
annoshift = 0.08
dendroshift = 0.02
new_y0 = pos.y0 + annoshift    # shift upwards
new_height = pos.height * 2
landmark_clustermap.ax_col_colors.set_position([pos.x0, new_y0, pos.width, new_height])
landmark_clustermap.ax_col_dendrogram.set_position([pos_d.x0, pos_d.y0 + annoshift + dendroshift, pos_d.width, pos_d.height])

# Move cluster names to top
landmark_clustermap.ax_heatmap.tick_params(right=False, top=True, labeltop=True, bottom=False, labelbottom=False, labelrotation=0, labelsize=14)

plt.close()
landmark_clustermap.savefig('./consensus_plots_postreview/LMNAT3.landmarkfeatures.v3.heatmap.medianplusEMD.022725.pdf', dpi=600)


In [35]:
# Make a variant venn diagram 
syn_z_cutoff = 2.5
p_value_thresh_tot = 10**(-2)/(np.prod(df_KSpvalues_merged.shape))
landmark_features = \
    ['Mean_NucleiExpanded_Intensity_MeanIntensity_CH1',
     'Mean_NuclearBoundary_Intensity_MeanIntensity_CH1',
     'Mean_NucleiExpanded_Granularity_1_CH1',
     'Mean_Nuclei_AreaShape_FormFactor'
    ]
feature_rename = \
    {
     'Mean_NucleiExpanded_Intensity_MeanIntensity_CH1':'Nuclear Intensity',
     'Mean_NuclearBoundary_Intensity_MeanIntensity_CH1':'Nuc Bndry Intensity',
     'Mean_NucleiExpanded_Granularity_1_CH1':'Nuc Granularity 1',
     'Mean_Nuclei_AreaShape_FormFactor': 'Nuc Circularity'
    }

# Get sets of significant variantsthat pass z-thresh and p-thresh
def find_significant_variants(df_z, df_p, features, p_thresh, z_thresh):
    # Because df_p is indexed by Variant, we can reindex or simply locate:
    # Make sure df_z rows align to df_p index
    df_p_filt = df_p.loc[df_z["Variant"]]
    
    sets_dict = {}
    for f in features:
        # Condition: p < p_thresh and EMD z > z_thresh
        pass_mask = (
            (df_p_filt[f].values <= p_thresh) &
            (df_z[f].values >= z_thresh)
        )
        # Collect the variants that pass
        passing_variants = df_p_filt[pass_mask].index
        sets_dict[f] = set(passing_variants)

    # get all missense set
    all_missense_set = set(df_z["Variant"])
    
    return sets_dict,all_missense_set

def sets_to_multiindex_series(all_variants, sets_dict, feature_ordering):
    """
    Convert a dictionary of sets {feature: set_of_variants} 
    plus a set of all variants into a multi-index Series for UpSetPlot.
    
    - all_variants: iterable of all Missense variants
    - sets_dict: {feature_name: set_of_variants_that_pass_that_feature}

    Returns:
    - A pd.Series with a multi-level boolean index (one level per feature),
      and values = count of how many variants fall into each True/False combination.
    """
    features = feature_ordering
    all_variants = sorted(all_variants)

    # 1) Build a boolean DataFrame
    df_bool = pd.DataFrame(False, index=all_variants, columns=features)
    for feat, varset in sets_dict.items():
        df_bool.loc[df_bool.index.intersection(varset), feat] = True

    # 2) Group by all columns to get counts of each combination
    #    This yields a multi-index (one level per column),
    #    with the aggregated values = how many rows (variants) in that combination.
    df_count = df_bool.groupby(features).size()

    return df_count

# set feature dataframes
df_z_toplot = \
    variant_EMD_merged[['Variant','Variant_Class'] + landmark_features]\
        .query('Variant_Class in ["Missense"]')\
        .query('Variant in @df_KSpvalues_merged.index')\
        .copy()
df_p_toplot = \
    df_KSpvalues_merged[landmark_features].copy()

# Rename columns
df_z_toplot.rename(columns=feature_rename, inplace=True)
df_p_toplot.rename(columns=feature_rename, inplace=True)

# Find significant variants
sig_sets,all_missense_set = \
    find_significant_variants(df_z_toplot, 
                              df_p_toplot, 
                              [feature_rename[f] for f in landmark_features], 
                              p_value_thresh_tot, 
                              syn_z_cutoff)

# Convert to Multi-Index set
df_count = sets_to_multiindex_series(all_missense_set, 
                                     sig_sets,
                                     feature_ordering=[feature_rename[f] for f in landmark_features])

# Pass that series to UpSet
pd.options.mode.copy_on_write = False # needed to fix upset plot, otherwise bug
upset_obj = UpSet(df_count,
                  sort_by='degree',
                  sort_categories_by='-input',
                  show_counts=True,
                  min_subset_size=10
                 )
upset_obj.style_subsets(
    present=['Nuclear Intensity'], 
    facecolor="teal"
)
upset_obj.style_subsets(
    present=['Nuc Bndry Intensity'], 
    facecolor="teal"
)
upset_obj.style_subsets(
    present=['Nuc Granularity 1'], 
    facecolor="fuchsia"
)
upset_obj.style_subsets(
    present=['Nuc Circularity'], 
    facecolor="chocolate"
)
upset_obj.plot()
pd.options.mode.copy_on_write = True

plt.ylabel('# Variants', fontsize=14)
plt.savefig('./consensus_plots_postreview/LMNAT3.upsetplot.landmarkfeatures.EMD.012625.pdf')
plt.close()


In [ ]:
## NEED SINGLE CELL DATA FOR THIS PORTION!
# Filter for variants in df_profiles_merged - need single cell image + feature data for this portion!
genotypes_df_filtered_r2 = \
    genotypes_df_r2\
        .query('aaChanges in @df_profiles_merged["Variant"] or aaChanges == "WT"')

# Filter well 1 variants
genotypes_df_filtered_r2_w1 = genotypes_df_filtered_r2.query('well == 1')


In [12]:
# Pick synonymous variants from clusters 2,5,6,7
sampled_synvariants_synclusters = (
    df_profiles_merged
        .query('Cluster in [2, 5, 6, 7]')
        .query('Variant_Class == "Synonymous"')
        .groupby('Cluster', group_keys=False)
        .sample(n=3, random_state=1337)
)
sampled_synvariants_synclusters[['Variant','Cluster']]

,Variant,Cluster
1395,L271L,2
1345,R196R,2
1145,Y211Y,2
1300,G232G,5
1012,A244A,5
1228,A242A,5
274,E262E,6
1111,D243D,6
55,E264E,6
1227,I229I,7


In [ ]:
# Visualize variant cells
variants_to_viz = \
    ['WT'] + list(sa
    mpled_synvariants_synclusters['Variant'].values)
variants_to_viz_cells = \
    visualize_cells_byvariant5(
        variants_to_viz, 
        genotypes_df_filtered_r2_w1.copy(),
        n_square=5,
        figure_size=10,
        pixel_size=0.0324,
        output_folder='./crops_cells_syn_v1/',
        clip_DAPI=(500,15000),
        clip_phenotype=(500,2000)
    )


In [ ]:
# Pick variants from every cluster other than 5,6,7
sampled_variants_pathclusters = (
    df_profiles_merged
        .merge(roc_auc_lmna, on='Variant', how='inner')
        .query('Cluster not in [5, 6, 7]')
        .query('Variant_Class == "Missense"')
        .groupby('Cluster', group_keys=False)
        .sample(n=5, random_state=1337)
)
sampled_variants_pathclusters[['Variant','Cluster']]

,Variant,Cluster
296,L241R,1
804,R240Y,1
400,E262I,1
318,Q258W,1
1758,Q251H,1
1201,K180P,2
980,E238C,2
246,Y267S,2
870,R196I,2
15,K180W,2


In [25]:
# Visualize variant cells
variants_to_viz = \
    ['WT'] + list(sampled_variants['Variant'].values)
variants_to_viz_cells = \
    visualize_cells_byvariant5(
        variants_to_viz, 
        genotypes_df_filtered_r2_w1.copy(),
        n_square=5,
        figure_size=10,
        pixel_size=0.0324,
        output_folder='./crops_cells_v4/',
        clip_DAPI=(500,15000),
        clip_phenotype=(500,2000)
    )


In [ ]:
## Pick variants from clusters 5,6,7
sampled_variants_WTlikeclusters = (
    df_profiles_merged
        .merge(roc_auc_lmna, on='Variant', how='inner')
        .query('Cluster in [5, 6, 7]')
        .query('Variant_Class == "Missense"')
        .groupby('Cluster', group_keys=False)
        .sample(n=5, random_state=1337)
)
sampled_variants_WTlikeclusters[['Variant','Cluster']]


,Variant,Cluster
798,R235Q,5
269,E257S,5
981,Q246V,5
983,E178R,5
996,V227T,5
87,K181A,6
913,A250M,6
993,E214T,6
379,E194V,6
964,E257G,6


In [126]:
# Visualize variant cells
variants_to_viz = \
    ['WT'] + list(sampled_variants['Variant'].values)
variants_to_viz_cells = \
    visualize_cells_byvariant5(
        variants_to_viz, 
        genotypes_df_filtered_r2_w1.copy(),
        n_square=5,
        figure_size=10,
        pixel_size=0.0324,
        output_folder='./crops_cells_v4/',
        clip_DAPI=(500,15000),
        clip_phenotype=(500,2000)
    )


In [128]:
# Visualize path variant cells
variants_to_viz = \
    ['E228V'] + list(clinvar_variants_filtersplice.query('`Variant Designation`.isin(["P","LP","P/LP"]) and Variant.isin(@df_profiles_merged["Variant"])')['Variant'].values)
variants_to_viz_cells = \
    visualize_cells_byvariant5(
        variants_to_viz, 
        genotypes_df_filtered_r2_w1.copy(),
        n_square=5,
        figure_size=10,
        pixel_size=0.0324,
        output_folder='./crops_cells_path_v2/',
        clip_DAPI=(500,15000),
        clip_phenotype=(500,2000)
    )


In [127]:
variants_to_viz = \
    list(clinvar_variants_filtersplice.query('`Variant Designation`.isin(["P","LP","P/LP"]) and Variant.isin(@df_profiles_merged["Variant"])')['Variant'].values)
df_profiles_merged.merge(roc_auc_lmna, on='Variant', how='left').query('Variant in @variants_to_viz')[['Variant','Cluster','AUC ROC']]


,Variant,Cluster,AUC ROC
222,Y267H,8,0.906254
257,E262K,8,0.741612
361,D230N,2,0.685651
465,G232E,9,0.878285
473,H222Y,2,0.637838
528,E203K,4,0.755228
694,E203G,8,0.787035
893,R190W,2,0.627447
1016,D192V,3,0.910530
1069,L204R,5,0.610009


In [138]:
# Show on UMAP
# --- user input ---
variants_to_label = \
    np.unique(
        list(sampled_variants_WTlikeclusters['Variant']) + \
        list(sampled_variants_pathclusters['Variant']) + \
        ['E228V'] + \
        list(clinvar_variants_filtersplice.query('`Variant Designation`.isin(["P","LP","P/LP"]) and Variant.isin(@df_profiles_merged["Variant"])')['Variant'].values)
    )
variant_col = "Variant"   # change if your column name is different

# Visualize Louvain Clusters
fig, ax = plt.subplots(figsize=(4, 4))
sns.scatterplot(
    data=df_profiles_merged,
    x="UMAP1",
    y="UMAP2",
    hue="Cluster",
    s=25,
    palette=sns.color_palette(),
    legend=False,
    ax=ax,
)

# Add labels for selected variants
df_lab = df_profiles_merged[df_profiles_merged[variant_col].isin(variants_to_label)].copy()

texts = []
for _, r in df_lab.iterrows():
    texts.append(
        ax.text(
            r["UMAP1"],
            r["UMAP2"],
            str(r[variant_col]),
            fontsize=10,          # tweak
            zorder=10,
        )
    )

# Repel labels (and optionally draw arrows to the points)
if len(texts) > 0:
    adjust_text(
        texts,
        ax=ax,
        expand_points=(1.4, 1.4),
        expand_text=(1.2, 1.2),
        force_points=0.15,
        force_text=0.5,
        lim=200,  # higher = more iterations if crowded
        arrowprops=dict(arrowstyle="-", lw=0.4, alpha=0.6),
        # If you want less arrow clutter, comment out arrowprops above.
    )

# Remove x/y ticks
ax.set_xticks([])
ax.set_yticks([])

# Labels
ax.set_xlabel("UMAP 1", fontsize=16)
ax.set_ylabel("UMAP 2", fontsize=16)

plt.tight_layout()
plt.close()

fig.savefig(
    "./consensus_plots_postreview/LMNAT3.UMAP.LouvainCluster.labeled.011826.pdf",
    dpi=600,
)


In [16]:
# Visualize cells by feature - landmark features
cell_crops_byfeature = \
    visualize_cells_byfeature3(
        landmark_features,
        genotypes_df_filtered_r2_w1.copy(),
        percentiles=(10,50,90),
        n_square=5,
        figure_size=10,
        axes_pad=0.2,
        pixel_size=0.0324,  # micrometers per pixel, example
        output_folder='./feature_crops_v4/',
        show_images=False,
        crops_col="CellCropsPath",
        masks_col="CellMasksPath",
        crop_index_col="CropIndex",
        variant_col="aaChanges",
        phenotype_channel=1,
        show_DAPI=True,
        dapi_channel=0,
        clip_DAPI=(500,15000),
        clip_phenotype=(500,2000),
        show_outline=True,
        outline_color='silver',
        outline_linewidth=1.2,
        mask_channel=0,
        plot_middle=None,
        random_state=0
    )


In [67]:
# Visualize cells by feature - radial distribution features
cell_crops_byfeature = \
    visualize_cells_byfeature3(
        ['Mean_NucleiExpanded_RadialDistribution_FracAtD_CH1_1of10',
         'Mean_NucleiExpanded_RadialDistribution_FracAtD_CH1_2of10',
         'Mean_NucleiExpanded_RadialDistribution_FracAtD_CH1_3of10',
         'Mean_NucleiExpanded_RadialDistribution_FracAtD_CH1_4of10',
         'Mean_NucleiExpanded_RadialDistribution_FracAtD_CH1_5of10',
         'Mean_NucleiExpanded_RadialDistribution_FracAtD_CH1_6of10',
         'Mean_NucleiExpanded_RadialDistribution_FracAtD_CH1_7of10',
         'Mean_NucleiExpanded_RadialDistribution_FracAtD_CH1_8of10',
         'Mean_NucleiExpanded_RadialDistribution_FracAtD_CH1_9of10',
         'Mean_NucleiExpanded_RadialDistribution_FracAtD_CH1_10of10'],
        genotypes_df_filtered_r2_w1.copy(),
        percentiles=(10,50,90),
        n_square=5,
        figure_size=10,
        axes_pad=0.2,
        pixel_size=0.0324,  # micrometers per pixel, example
        output_folder='./feature_crops_radialdistributionfracatD_v1/',
        show_images=False,
        crops_col="CellCropsPath",
        masks_col="CellMasksPath",
        crop_index_col="CropIndex",
        variant_col="aaChanges",
        phenotype_channel=1,
        show_DAPI=True,
        dapi_channel=0,
        clip_DAPI=(500,15000),
        clip_phenotype=(500,2000),
        show_outline=True,
        outline_color='silver',
        outline_linewidth=1.2,
        mask_channel=0,
        plot_middle=None,
        random_state=0
    )


In [54]:
# Visualize cells by feature - granularity features
cell_crops_byfeature = \
    visualize_cells_byfeature3(
        ['Mean_NucleiExpanded_Granularity_1_CH1',
         'Mean_NucleiExpanded_Granularity_2_CH1',
         'Mean_NucleiExpanded_Granularity_3_CH1',
         'Mean_NucleiExpanded_Granularity_4_CH1',
         'Mean_NucleiExpanded_Granularity_5_CH1',
         'Mean_NucleiExpanded_Granularity_6_CH1',
         'Mean_NucleiExpanded_Granularity_7_CH1',
         'Mean_NucleiExpanded_Granularity_8_CH1'],
        genotypes_df_filtered_r2_w1.copy(),
        percentiles=(10,50,90),
        n_square=5,
        figure_size=10,
        axes_pad=0.2,
        pixel_size=0.0324,  # micrometers per pixel, example
        output_folder='./feature_crops_granularity_v1/',
        show_images=False,
        crops_col="CellCropsPath",
        masks_col="CellMasksPath",
        crop_index_col="CropIndex",
        variant_col="aaChanges",
        phenotype_channel=1,
        show_DAPI=True,
        dapi_channel=0,
        clip_DAPI=(500,15000),
        clip_phenotype=(500,2000),
        show_outline=True,
        outline_color='silver',
        outline_linewidth=1.2,
        mask_channel=0,
        plot_middle=None,
        random_state=0
    )
